# 05. Safe RL: Constrained MDP, PPO-Lagrangian, CVaR, and Control Barrier Functions

- **Category:** 11_ai_safety_privacy_and_verification
- **Lineage & Citations:** Altman (1999 *Constrained Markov Decision Processes*); Chow et al. (2015 *Risk-Sensitive and Robust Decision-Making: CVaR*); Ray et al. (2019 *Benchmarking Safe Exploration in Deep Reinforcement Learning*); Ames et al. (2019 *Control Barrier Functions: Theory and Applications*).
- **Purpose:** Formulate and evaluate safe reinforcement learning mechanisms: Constrained MDPs via PPO-Lagrangian primal-dual optimization, Conditional Value-at-Risk (CVaR) tail-risk metrics, and real-time Control Barrier Function (CBF-QP) safety filters.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [04. Abliteration Refusal Surgery Circuit Breakers and Sleeper Agents](04_abliteration_refusal_surgery_circuit_breakers_and_sleeper_agents.ipynb) $\to$ `🔗 Downstream Syntheses:` [06. Backdoor Trojan and Poisoning Defense Neural Cleanse and Spectral](06_backdoor_trojan_and_poisoning_defense_neural_cleanse_and_spectral.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use naive static reward penalties $R - c \cdot \text{Cost}$ for hard safety guarantees (cannot satisfy non-convex constraint boundaries); do not omit actuator saturation limits in CBF-QP filters.

## Mathematical Foundations & Governing Equations

1. **PPO-Lagrangian Primal Objective**: $\mathcal{L}_\pi = -\mathbb{E}[\min(r A_R, \text{clip}(r) A_R)] + \lambda \cdot \mathbb{E}[r A_C]$
2. **Dual Multiplier Ascent**: $\lambda' = \max(0, \lambda + \eta_\lambda (J_C - d_{\max}))$
3. **Tail-Risk Measure**: $\text{CVaR}_\alpha(C) = \mathbb{E}[C \mid C \ge \text{VaR}_\alpha(C)]$
4. **CBF-QP Forward-Invariant Filter**: $u^* = u_{\text{nom}} + \frac{\max(0, -(\nabla h^\top u_{\text{nom}} + \alpha h))}{\|\nabla h\|_2^2 + \epsilon} \nabla h$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/11_ai_safety_privacy_and_verification/05_safe_rl_constrained_mdp_ppo_lagrangian_cvar_and_cbf.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def calc_ppo_lag(lp, olp, ar, ac, lam=1.2, eps=0.2):
    ratio = torch.exp(lp - olp)
    surr1 = ratio * ar
    surr2 = torch.clamp(ratio, 1.0 - eps, 1.0 + eps) * ar
    return -torch.min(surr1, surr2).mean() + float(lam) * (ratio * ac).mean()

def calc_lam_step(lam, cost_m, cost_lim, lr=0.1):
    return max(0.0, float(lam) + float(lr) * (float(cost_m) - float(cost_lim)))

def calc_cvar(costs, alpha=0.25):
    c_sort, _ = torch.sort(costs.view(-1))
    k = max(1, int(round(float(alpha) * c_sort.shape[0])))
    return torch.mean(c_sort[-k:])

def calc_cbf_qp(u_nom, grad_h, h_val, alpha_cbf=1.5):
    b = float(alpha_cbf) * h_val.view(-1)
    c_viol = torch.sum(grad_h * u_nom, dim=-1) + b
    deficit = torch.clamp(-c_viol, min=0.0)
    corr = deficit / (torch.sum(grad_h * grad_h, dim=-1) + 1e-8)
    return u_nom + corr.unsqueeze(-1) * grad_h

In [ ]:
# Parity verification against golden reference tensors
ppo_loss = calc_ppo_lag(weights["logp"], weights["old_logp"], weights["adv_reward"], weights["adv_cost"], 1.2, 0.2)
assert torch.max(torch.abs(ppo_loss - weights["ppo_lag_ref"])) < 1e-5

lam_up = calc_lam_step(1.2, 0.35, 0.20, 0.1)
assert abs(lam_up - weights["lam_updated_ref"].item()) < 1e-5

cvar = calc_cvar(weights["costs"], 0.25)
assert torch.max(torch.abs(cvar - weights["cvar_ref"])) < 1e-5

u_safe = calc_cbf_qp(weights["u_nominal"], weights["grad_h"], weights["h_val"], 1.5)
assert torch.max(torch.abs(u_safe - weights["u_safe_ref"])) < 1e-5
print("All PPO-Lagrangian, CVaR, and CBF-QP safety assertions passed!")

In [ ]:
# Benchmarking Safe RL operators
t0 = time.perf_counter()
for _ in range(500):
    _ = calc_ppo_lag(weights["logp"], weights["old_logp"], weights["adv_reward"], weights["adv_cost"], 1.2, 0.2)
    _ = calc_cvar(weights["costs"], 0.25)
    _ = calc_cbf_qp(weights["u_nominal"], weights["grad_h"], weights["h_val"], 1.5)
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Combined Safe RL (PPO-Lag + CVaR + CBF-QP) latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: Unconstrained Action Violating Safety vs Instantaneous CBF-QP Correction
# Unconstrained agent commands forward acceleration heading into an obstacle (grad_h = [-1.0, 0.0])
u_risky = torch.tensor([[2.5, 0.5]])  # High positive speed
grad_obs = torch.tensor([[-1.0, 0.0]])  # Barrier gradient points away from obstacle
h_dist = torch.tensor([-0.2])  # Critical proximity: h(x) is dangerously close
alpha = 2.0

# Nominal action violation condition: grad_h . u + alpha * h
c_nom = (grad_obs * u_risky).sum().item() + alpha * h_dist.item()  # (-2.5) + (-0.4) = -2.9 < 0 (VIOLATION)
u_filtered = calc_cbf_qp(u_risky, grad_obs, h_dist, alpha_cbf=alpha)
c_filtered = (grad_obs * u_filtered).sum().item() + alpha * h_dist.item()  # Exactly 0.0 (SAFE)

print(f"Barrier condition: Nominal = {c_nom:.3f} (UNSAFE) -> Filtered = {c_filtered:.4e} (SAFE)")
print(f"Action corrected from {u_risky.tolist()[0]} to {u_filtered.tolist()[0]}")
assert c_nom < -2.0, "Nominal control severely violates barrier condition!"
assert abs(c_filtered) < 1e-6, "CBF-QP filter must guarantee exact non-negative barrier compliance!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **PPO-Lagrangian** solves CMDPs via primal-dual optimization, dynamically adapting constraint penalties without manual reward tuning.
2. **CVaR Tail-Risk** penalizes catastrophic rare trajectories rather than risk-neutral expected values.
3. **Control Barrier Functions (CBF-QP)** guarantee provable forward invariance of safe state sets in real-time closed-loop control.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [05_safe_rl_constrained_mdp_ppo_lagrangian_cvar_and_cbf.md](../../markdown/11_ai_safety_privacy_and_verification/05_safe_rl_constrained_mdp_ppo_lagrangian_cvar_and_cbf.md)
- **PyTorch 2.x Module:** [05_safe_rl_constrained_mdp_ppo_lagrangian_cvar_and_cbf.py](../../pytorch2/11_ai_safety_privacy_and_verification/05_safe_rl_constrained_mdp_ppo_lagrangian_cvar_and_cbf.py)
- **Functional JAX Module:** [05_safe_rl_constrained_mdp_ppo_lagrangian_cvar_and_cbf.py](../../jax/11_ai_safety_privacy_and_verification/05_safe_rl_constrained_mdp_ppo_lagrangian_cvar_and_cbf.py)
- **Native Rust Candle Track:** [05_safe_rl_constrained_mdp_ppo_lagrangian_cvar_and_cbf.rs](../../candle/11_ai_safety_privacy_and_verification/05_safe_rl_constrained_mdp_ppo_lagrangian_cvar_and_cbf.rs)
- **Hardware Kernel (CPU / WGSL):** [05_safe_rl_constrained_mdp_ppo_lagrangian_cvar_and_cbf.rs](../../kernels/11_ai_safety_privacy_and_verification/05_safe_rl_constrained_mdp_ppo_lagrangian_cvar_and_cbf.rs)
- **Golden SafeTensors:** [05_safe_rl_constrained_mdp_ppo_lagrangian_cvar_and_cbf.safetensors](../../weights/11_ai_safety_privacy_and_verification/05_safe_rl_constrained_mdp_ppo_lagrangian_cvar_and_cbf.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 04. Abliteration Refusal Surgery Circuit Breakers and Sleeper Agents](04_abliteration_refusal_surgery_circuit_breakers_and_sleeper_agents.ipynb) | [Master Index](../../README.md#11_ai_safety_privacy_and_verification) | [Next: 06. Backdoor Trojan and Poisoning Defense Neural Cleanse and Spectral](06_backdoor_trojan_and_poisoning_defense_neural_cleanse_and_spectral.ipynb) &rarr;